# This notebook contains code snippets for analyzing UQ and SA data generated by the UQEF-Dynamic Framework for the HBV-SASK Hydro Model.

# Import Libraries

In [ ]:
import os
import dill
import numpy as np
import sys
import pathlib
import pandas as pd
import pickle
import time
from collections import defaultdict

import scipy.special


In [ ]:
# importing modules/libs for plotting
from plotly.subplots import make_subplots
import plotly.graph_objects as go
import plotly.express as px

# from plotly.offline import plot
import plotly.offline as pyo
# Set notebook mode to work in offline
pyo.init_notebook_mode()

import matplotlib.pyplot as plt

pd.options.plotting.backend = "plotly"

In [ ]:
import chaospy as cp
import uqef
print(cp.__version__)

In [ ]:
# TODO - change these paths accordingly
# sys.path.insert(1, '/work/ga45met/Hydro_Models/HBV-SASK-py-tool')
sys.path.insert(1, '/work/ga45met/mnt/linux_cluster_2/UQEF-Dynamic')

from uqef_dynamic.utils import utility
from uqef_dynamic.utils import uqef_dynamic_utils
from uqef_dynamic.models.hbv_sask import hbvsask_utility as hbv
from uqef_dynamic.models.hbv_sask import HBVSASKModel as hbvmodel
from uqef_dynamic.models.hbv_sask import HBVSASKStatistics as HBVSASKStatistics
from uqef_dynamic.utils import create_stat_object


# Defining paths

In [ ]:
# TODO - change these paths accordingly
hbv_model_data_path = pathlib.Path("/work/ga45met/Hydro_Models/HBV-SASK-data")
inputModelDir = hbv_model_data_path

In [ ]:
# TODO - change these paths accordingly
basis_workingDir = pathlib.Path('/work/ga45met/paper_uqef_dynamic_sim/hbvsask_runs_lxc_autumn_24')
# 10 MC gPCE p=5 CT0.7 150 000 Random 2006-07 Q_cms; AET
workingDir = basis_workingDir / "hbv_uq_cm4.0094"


In [ ]:
directory_for_saving_plots = pathlib.Path('/work/ga45met/paper_uqef_dynamic_sim/hbv_sask/mc_gpce_p5_150000_ct07_2006_07_oldman')

if not str(directory_for_saving_plots).endswith("/"):
    directory_for_saving_plots = str(directory_for_saving_plots) + "/"
directory_for_saving_plots =  pathlib.Path(directory_for_saving_plots)


# Reading Saved Files

In [ ]:
args_files = utility.get_dict_with_output_file_paths_based_on_workingDir(
    workingDir,
)

for key, value in args_files.items():
    globals()[key] = value

In [ ]:
with open(args_file, 'rb') as f:
    uqsim_args = pickle.load(f)
uqsim_args_dict = vars(uqsim_args)
model = uqsim_args_dict["model"]
inputModelDir = uqsim_args_dict["inputModelDir"]
uqsim_args_dict

In [ ]:
print(configuration_object_file)
with open(configuration_object_file, 'rb') as f:
    configurationObject = dill.load(f)
configurationObject

In [ ]:
basis = configurationObject['model_settings']['basis']
print(f"basin {basis}")

In [ ]:
simulation_settings_dict = utility.read_simulation_settings_from_configuration_object(configurationObject)
simulation_settings_dict


#### extra anlyzing

In [ ]:
# Number of uncertain parameters
len(configurationObject["parameters"])

In [ ]:
# Quantity of Interes
print(configurationObject["simulation_settings"]["qoi"])
print(configurationObject["simulation_settings"]["qoi_column"])

# Reading Nodes and Parameters

In [ ]:
with open(nodes_file, 'rb') as f:
#     simulationNodes = dill.load(f)
    simulationNodes = pickle.load(f)
simulationNodes

print(simulationNodes.nodes.shape)
print(simulationNodes.parameters.shape)

#### extra anlyzing

In [ ]:
simulationNodes.nodes

In [ ]:
simulationNodes.nodes.shape

In [ ]:
simulationNodes.distNodes.shape

In [ ]:
simulationNodes.parameters

In [ ]:
simulationNodes.parameters.shape

In [ ]:
simulationNodes.joinedDists

In [ ]:
simulationNodes.joinedStandardDists

# Reading Parameters and GoF Computed Data

In [ ]:
if df_index_parameter_file.is_file():
    df_index_parameter = pd.read_pickle(df_index_parameter_file, compression="gzip")
else:
    df_index_parameter = None
df_index_parameter

In [ ]:
if df_index_parameter is not None:
    params_list = utility._get_parameter_columns_df_index_parameter_gof(
        df_index_parameter)
else:
    params_list = []
    for single_param in configurationObject["parameters"]:
        params_list.append(single_param["name"])
params_list

In [ ]:
if df_index_parameter_gof_file.is_file():
    df_index_parameter_gof = pd.read_pickle(df_index_parameter_gof_file, compression="gzip")
    df_index_parameter_gof
else:
    print(f"Be careful - {df_index_parameter_gof_file} does not exist!")
    df_index_parameter_gof = None
df_index_parameter_gof

In [ ]:
if df_index_parameter_gof is not None:
    gof_list = utility._get_gof_columns_df_index_parameter_gof(
        df_index_parameter_gof)
else:
    gof_list = None
    print(f"Be careful - {df_index_parameter_gof_file} does not exist - therefore gof_list is not populated!")
gof_list

#### extra anlyzing

In [ ]:
df_index_parameter_gof[gof_list].describe(include=np.number)

In [ ]:
fig = utility.plot_2d_matrix_static_from_list(df_index_parameter[params_list], title="Plot 2D projected positions of the nodes")
filename = directory_for_saving_plots / "pairplot_simulation_nodes.pdf"
fig.savefig(str(filename), format="pdf")

In [ ]:
# one can as well call uqef_dynamic_utils.gof_values_GaussianKDE to produce this plot

fig, axs = plt.subplots(1, len(gof_list), figsize=(20, 10))

for i in range(len(gof_list)):
    single_gof = gof_list[i]
    min_single_gof = df_index_parameter_gof[single_gof].min() - abs(df_index_parameter_gof[single_gof].min())*0.001
    max_single_gof = df_index_parameter_gof[single_gof].max() + abs(df_index_parameter_gof[single_gof].max())*0.001
    t = np.linspace(min_single_gof, max_single_gof, 1000)
    gof_eval = df_index_parameter_gof[single_gof].values
    distribution = cp.GaussianKDE(gof_eval, h_mat=0.005 ** 2)
    axs[i,].hist(gof_eval, bins=100, density=True, alpha=0.5)
    axs[i,].plot(t, distribution.pdf(t), label=f"KDE {single_gof}")
    plt.setp(axs[i,], xlabel=f'{single_gof}')
    axs[i,].grid()
plt.setp(axs[0], ylabel='PDF')
# fig.suptitle(f'{basis}; QoI:Q_cms', fontsize=16)
filename = directory_for_saving_plots / "dist_of_gofs.pdf"
fig.tight_layout()
fig.savefig(str(filename), format="pdf")
plt.show()


In [ ]:
# one can as well call uqef_dynamic_utils.gof_values_GaussianKDE to produce this plot

fig, axs = plt.subplots(1, len(gof_list), figsize=(20, 10))

for i in range(len(gof_list)):
    single_gof = gof_list[i]
    min_single_gof = df_index_parameter_gof[single_gof].min() - abs(df_index_parameter_gof[single_gof].min())*0.001
    max_single_gof = df_index_parameter_gof[single_gof].max() + abs(df_index_parameter_gof[single_gof].max())*0.001
    t = np.linspace(min_single_gof, max_single_gof, 1000)
    gof_eval = df_index_parameter_gof[single_gof].values
    distribution = cp.GaussianKDE(gof_eval, h_mat=0.005 ** 2)
    axs[i,].plot(t, distribution.cdf(t), label=f"KDE {single_gof}")
    plt.setp(axs[i,], xlabel=f'{single_gof}')
    axs[i,].grid()
plt.setp(axs[0], ylabel='CDF')
# fig.suptitle(f'{basis} Basin; QoI:Q_cms', fontsize=16)
filename = directory_for_saving_plots / "cdf_of_gofs.pdf"
fig.tight_layout()
fig.savefig(str(filename), format="pdf")
plt.show()


In [ ]:
df_index_parameter_gof['RMSE'].min()

In [ ]:
fig = utility.plot_subplot_params_hist_from_df(df_index_parameter_gof)
# fig.update_layout(title="Prior Distribution of the Parameters",)
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=50,  # Left margin
        r=10   # Right margin
    )
)
plot_filename = directory_for_saving_plots / f"prior_dist_of_params.pdf"
fig.write_image(str(plot_filename), format="pdf", width=1200,)
fig.show()

In [ ]:
df_index_parameter_gof["TT"].nunique()

In [ ]:
df_index_parameter_gof["TT"].unique()

In [ ]:
print(df_index_parameter_gof["K1"].nunique())
print(df_index_parameter_gof["K1"].unique())

In [ ]:
print(df_index_parameter_gof['RMSE'].min())
print(df_index_parameter_gof['RMSE'].max())
print(df_index_parameter_gof['LogNSE'].min())
print(df_index_parameter_gof['LogNSE'].max())
print(df_index_parameter_gof['NSE'].min())
print(df_index_parameter_gof['NSE'].max())
print(df_index_parameter_gof['KGE'].min())
print(df_index_parameter_gof['KGE'].max())

In [ ]:
df_index_parameter_gof[df_index_parameter_gof['NSE']==df_index_parameter_gof['NSE'].max()]#[params_list]

In [ ]:
df_index_parameter_gof[df_index_parameter_gof['RMSE']==df_index_parameter_gof['RMSE'].min()]#[params_list]

In [ ]:
df_index_parameter_gof[df_index_parameter_gof['KGE']==df_index_parameter_gof['KGE'].max()]#[params_list]

In [ ]:
df_index_parameter_gof[df_index_parameter_gof['LogNSE']==df_index_parameter_gof['LogNSE'].max()]#[params_list]

In [ ]:
fig = utility.plot_subplot_params_hist_from_df_conditioned(
    df_index_parameter_gof, name_of_gof_column="NSE", 
    threshold_gof_value = 0.6, comparison="greater")
fig.update_layout(title="\
Parameter Distribution Conditioned on: values of NSE greater than 0.6",)
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=50,  # Left margin
        r=10   # Right margin
    )
)
plot_filename = directory_for_saving_plots / f"dist_of_params_nse_greater_than_06.pdf"
fig.write_image(str(plot_filename), format="pdf", width=1200,)
fig.show()

In [ ]:
fig = utility.plot_subplot_params_hist_from_df_conditioned(
    df_index_parameter_gof, name_of_gof_column="NSE", 
    threshold_gof_value = 0.2, comparison="greater")
fig.update_layout(title="\
Parameter Distribution Conditioned on: values of NSE greater than 0.2",)
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=50,  # Left margin
        r=10   # Right margin
    )
)
plot_filename = directory_for_saving_plots / f"dist_of_params_nse_greater_than_02.pdf"
fig.write_image(str(plot_filename), format="pdf", width=1200,)
fig.show()

In [ ]:
fig = utility.plot_subplot_params_hist_from_df_conditioned(
    df_index_parameter_gof, name_of_gof_column="LogNSE", 
    threshold_gof_value = 0.0, comparison="greater")
fig.update_layout(title="\
Parameter Distribution Conditioned on: values of LogNSE greater than 0.0",)
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=50,  # Left margin
        r=10   # Right margin
    )
)
plot_filename = directory_for_saving_plots / f"dist_of_params_lognse_greater_than_0.pdf"
fig.write_image(str(plot_filename), format="pdf", width=1200,)
fig.show()

In [ ]:
# Plotting Conditional Marginal CDF of single parameter
single_param = "TT"
single_gof = "NSE"
threshold_gof_value = 0.6
mask = df_index_parameter_gof[single_gof] > threshold_gof_value
df_index_parameter_gof_subset = df_index_parameter_gof[mask]

min_single_param = df_index_parameter_gof[single_param].min() - abs(df_index_parameter_gof[single_param].min())*0.001
max_single_param = df_index_parameter_gof[single_param].max() + abs(df_index_parameter_gof[single_param].max())*0.001
t = np.linspace(min_single_param, max_single_param, 1000)
single_param_eval = df_index_parameter_gof_subset[single_param].values
distribution = cp.GaussianKDE(single_param_eval, h_mat=0.005 ** 2)
plt.plot(t, distribution.cdf(t), label=f"CM CDF - {single_param} conditiond on {single_gof} greater than {threshold_gof_value}")
plt.legend()
filename = directory_for_saving_plots / "cdf_tt_marginal_cdf_nse_greater_than_06.pdf"
plt.tight_layout()
plt.savefig(str(filename), format="pdf")
plt.show()

In [ ]:
# Plotting Conditional Marginal CDF of single parameter
single_param = "C0"
single_gof = "NSE"
threshold_gof_value = 0.6
mask = df_index_parameter_gof[single_gof] > threshold_gof_value
df_index_parameter_gof_subset = df_index_parameter_gof[mask]

min_single_param = df_index_parameter_gof[single_param].min() - abs(df_index_parameter_gof[single_param].min())*0.001
max_single_param = df_index_parameter_gof[single_param].max() + abs(df_index_parameter_gof[single_param].max())*0.001
t = np.linspace(min_single_param, max_single_param, 1000)
single_param_eval = df_index_parameter_gof_subset[single_param].values
distribution = cp.GaussianKDE(single_param_eval, h_mat=0.005 ** 2)
plt.plot(t, distribution.cdf(t), label=f"CM CDF - {single_param} conditiond on {single_gof} greater than {threshold_gof_value}")
plt.legend()
filename = directory_for_saving_plots / "cdf_c0_marginal_cdf_nse_greater_than_06.pdf"
plt.tight_layout()
plt.savefig(str(filename), format="pdf")
plt.show()

In [ ]:
fig = utility.plot_subplot_params_hist_from_df_conditioned(
    df_index_parameter_gof, name_of_gof_column="LogNSE", 
    threshold_gof_value = 0.6, comparison="greater")
fig.update_layout(title="\
Parameter Distribution Conditioned on: values of LogNSE greater than 0.6",)
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=50,  # Left margin
        r=10   # Right margin
    )
)
plot_filename = directory_for_saving_plots / f"dist_of_params_lognse_greater_than_06.pdf"
fig.write_image(str(plot_filename), format="pdf", width=1200,)
fig.show()

In [ ]:
fig = utility.plot_subplot_params_hist_from_df_conditioned(
    df_index_parameter_gof, name_of_gof_column="RMSE", 
    threshold_gof_value = 20.0, comparison="smaller")
fig.update_layout(title="\
Parameter Distribution Conditioned on: values of RMSE smaller than 20",)
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=50,  # Left margin
        r=10   # Right margin
    )
)
plot_filename = directory_for_saving_plots / f"dist_of_params_rmse_less_than_20.pdf"
fig.write_image(str(plot_filename), format="pdf", width=1200,)
fig.show()

In [ ]:
fig = utility.plot_subplot_params_hist_from_df_conditioned(
    df_index_parameter_gof, name_of_gof_column="KGE", 
    threshold_gof_value = 0.6, comparison="greater")
fig.update_layout(title="\
Parameter Distribution Conditioned on: values of KGE greater than 0.6",)
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=50,  # Left margin
        r=10   # Right margin
    )
)
plot_filename = directory_for_saving_plots / f"dist_of_params_kge_greater_than_06.pdf"
fig.write_image(str(plot_filename), format="pdf", width=1200,)
fig.show()

In [ ]:
# TODO - does not work after updating
fig = utility.plot_scatter_matrix_params_vs_gof(
    df_index_parameter_gof, name_of_gof_column="RMSE",
    hover_name="index_run", columns_with_parameters=params_list
)
fig.show()

In [ ]:
fig = utility.plot_parallel_params_vs_gof(
    df_index_parameter_gof, name_of_gof_column="RMSE", list_of_params=params_list
)
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=50,  # Top margin
        b=20,  # Bottom margin
        l=20,  # Left margin
        r=20   # Right margin
    )
)
plot_filename = directory_for_saving_plots / f"parallel_plot_params_vs_rmse.pdf"
fig.write_image(str(plot_filename), format="pdf", width=1200,)
fig.show()

In [ ]:
fig = utility.plot_parallel_params_vs_gof(
    df_index_parameter_gof, name_of_gof_column="NSE", list_of_params=params_list
)
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=50,  # Top margin
        b=20,  # Bottom margin
        l=50,  # Left margin
        r=20   # Right margin
    )
)
plot_filename = directory_for_saving_plots / f"parallel_plot_params_vs_nse.pdf"
fig.write_image(str(plot_filename), format="pdf",  width=1200,)
fig.show()

In [ ]:
fig = utility.plot_parallel_params_vs_gof(
    df_index_parameter_gof, name_of_gof_column="KGE", list_of_params=params_list
)
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=50,  # Top margin
        b=20,  # Bottom margin
        l=50,  # Left margin
        r=20   # Right margin
    )
)
plot_filename = directory_for_saving_plots / f"parallel_plot_params_vs_kge.pdf"
fig.write_image(str(plot_filename), format="pdf",  width=1200,)
fig.show()

## Nodes & Parameter  in a DataFrame -  after transformation

In [ ]:
df_nodes = utility.get_df_from_simulationNodes(simulationNodes, nodes_or_paramters="nodes", params_list=params_list)
df_nodes_params = utility.get_df_from_simulationNodes(simulationNodes, nodes_or_paramters="parameters",  params_list=params_list)

#### extra anlyzing

In [ ]:
df_nodes

In [ ]:
df_nodes['alpha'].values

In [ ]:
df_nodes_params

# Reading Saved Simulations
Note: This migh be a huge file, especially for MC/Saltelli kind of simulations

In [ ]:
# or in case of a big simulation, skip reading df_simulation_result
read_all_saved_simulations_file = False
if read_all_saved_simulations_file and df_simulations_file.is_file():
    # Reading Saved Simulations - Note: This migh be a huge file,
    # especially for MC/Saltelli kind of simulations
    df_simulation_result = pd.read_pickle(df_simulations_file, compression="gzip")
else:
    df_simulation_result = None
df_simulation_result

# Re-create Statistics Object and DataFrame Object That contains all the Statistics Data

In [ ]:
statisticsObject = create_stat_object.create_statistics_object(
    configurationObject, uqsim_args_dict, workingDir, model=model)


In [ ]:
# uqsim_args_dict['instantly_save_results_for_each_time_step'] = False  # Whatch-out; this is required just sometimes...

statistics_dictionary = uqef_dynamic_utils.read_all_saved_statistics_dict(\
    workingDir=workingDir, 
    list_qoi_column=statisticsObject.list_qoi_column, 
    single_timestamp_single_file=uqsim_args_dict.get("instantly_save_results_for_each_time_step", False), 
    throw_error=True
)

### Once you have satistics_dictionary extend StatisticsObject...

In [ ]:
# just to check...
print(statisticsObject.list_qoi_column)
print(type(statisticsObject.pdTimesteps))

In [ ]:
uqef_dynamic_utils.extend_statistics_object(
    statisticsObject=statisticsObject, 
    statistics_dictionary=statistics_dictionary, 
    df_simulation_result=df_simulation_result,  # df_simulation_result=None,
    get_measured_data=False, 
    get_unaltered_data=False
)

# Add measured Data
# This is hardcoded for HBV-inputModelDir_basis has to be overwritten because it point to the path on LinuxCluster
statisticsObject.inputModelDir_basis = hbv_model_data_path / basis
statisticsObject.inputModelDir_basis
statisticsObject.get_measured_data(
    timestepRange=(statisticsObject.timesteps_min, statisticsObject.timesteps_max), 
    qoi_column_name='Q_cms',
    transforme_mesured_data_as_original_model="False")

# Create a Pandas.DataFrame
df_statistics = statisticsObject.create_df_from_statistics_data()

# Add forcing Data
statisticsObject.get_forcing_data(time_column_name="TimeStamp")

# Merge Everything
df_statistics_and_measured = pd.merge(
    statisticsObject.df_statistics, statisticsObject.forcing_df, left_on=statisticsObject.time_column_name, right_index=True)
print(df_statistics_and_measured)

df_statistics_and_measured[utility.TIME_COLUMN_NAME] = pd.to_datetime(df_statistics_and_measured[utility.TIME_COLUMN_NAME])
df_statistics_and_measured = df_statistics_and_measured.sort_values(by=utility.TIME_COLUMN_NAME)

In [ ]:
statisticsObject.qoi

In [ ]:
if isinstance(statisticsObject.qoi, list):
    single_time_stampe = list(statisticsObject.result_dict[statisticsObject.qoi[0]].keys())[0]
    generalized_sobol_total_index_TT = statisticsObject.result_dict[statisticsObject.qoi[0]][single_time_stampe]['generalized_sobol_total_index_TT']
else:
    single_time_stampe = list(statisticsObject.result_dict[statisticsObject.qoi].keys())[0]
    generalized_sobol_total_index_TT = statisticsObject.result_dict[statisticsObject.qoi][single_time_stampe]['generalized_sobol_total_index_TT']  
print(f"single_time_stampe-{single_time_stampe} - generalized_sobol_total_index_TT-{generalized_sobol_total_index_TT}")

In [ ]:
statisticsObject.df_statistics.columns

In [ ]:
df_statistics_and_measured['Sobol_m_TT'].mean()

### trying out different thing

In [ ]:
print(len(set((statisticsObject.result_dict[statisticsObject.list_qoi_column[0]].keys()))))

assert set(statisticsObject.pdTimesteps) == set(statistics_dictionary[statisticsObject.list_qoi_column[0]].keys()), "Hmmmm"

print(statisticsObject.list_qoi_column)

print(list(statistics_dictionary.keys()))

print(list(statistics_dictionary[list(statistics_dictionary.keys())[0]].keys()))

print(statisticsObject.timesteps_min)
print(list(statistics_dictionary.keys()))

# print(statistics_dictionary[statisticsObject.list_qoi_column[0]][pd.Timestamp('2007-04-30 00:00:00')].keys())

In [ ]:
df_statistics_and_measured.columns

In [ ]:
df_statistics_and_measured.shape

In [ ]:
statisticsObject.df_statistics.shape

# Describing Statistics Data

### Note: these are two relevant DFs which do not have to be identical necessarily
### df_statistics_and_measured and statisticsObject.df_statistics

In [ ]:
df_statistics_and_measured['qoi'].unique()

In [ ]:
# Note: this works on df_statistics_and_measured
# This does not make much sense when there is more QoIs
for sinlge_qoi in list(df_statistics_and_measured['qoi'].unique()):
    print(f"Description - {sinlge_qoi}")
    df_statistics_and_measured_subset = df_statistics_and_measured[df_statistics_and_measured['qoi']==sinlge_qoi]
    print(df_statistics_and_measured_subset.describe(include=np.number))

In [ ]:
# the same as above - just convenient in multi QoIs set-up
# Note: this works on statisticsObject.df_statistics
statisticsObject.describe_df_statistics()

In [ ]:
# Examing df_statistics_and_measured
df_statistics_and_measured[df_statistics_and_measured["precipitation"]>0]
df_statistics_and_measured[df_statistics_and_measured["E"]<0]

# Plotting different time-series

In [ ]:
df_statistics_and_measured.columns

In [ ]:
# Note: from this point on df_statistics_and_measured differs from statisticsObject.df_statistics
set_mean_prediction_to_zero = True
set_lower_predictions_to_zero = True

if 'StdDev' not in df_statistics_and_measured.columns and 'Var' in df_statistics_and_measured.columns:
    df_statistics_and_measured["StdDev"] = np.sqrt(df_statistics_and_measured['Var'])
    
if 'StdDev' in df_statistics_and_measured.columns:
    if "E_minus_std" not in df_statistics_and_measured.columns and "E_plus_std" not in df_statistics_and_measured.columns:
        df_statistics_and_measured["E_minus_std"] = df_statistics_and_measured['E'] - df_statistics_and_measured['StdDev']
        df_statistics_and_measured["E_plus_std"] = df_statistics_and_measured['E'] + df_statistics_and_measured['StdDev']
    if "E_minus_2std" not in df_statistics_and_measured.columns and "E_plus_2std" not in df_statistics_and_measured.columns:
        df_statistics_and_measured["E_minus_2std"] = df_statistics_and_measured['E'] - 2*df_statistics_and_measured['StdDev']
        df_statistics_and_measured["E_plus_2std"] = df_statistics_and_measured['E'] + 2*df_statistics_and_measured['StdDev']
elif 'Var' in df_statistics_and_measured.columns:
    if "E_minus_std" not in df_statistics_and_measured.columns and "E_plus_std" not in df_statistics_and_measured.columns:
        df_statistics_and_measured["E_minus_std"] = df_statistics_and_measured['E'] - np.sqrt(df_statistics_and_measured['Var'])
        df_statistics_and_measured["E_plus_std"] = df_statistics_and_measured['E'] + np.sqrt(df_statistics_and_measured['Var'])
        df_statistics_and_measured['E_minus_std'] = df_statistics_and_measured['E_minus_std'].apply(lambda x: max(0, x))
    if "E_minus_2std" not in df_statistics_and_measured.columns and "E_plus_2std" not in df_statistics_and_measured.columns:
        df_statistics_and_measured["E_minus_2std"] = df_statistics_and_measured['E'] - 2*np.sqrt(df_statistics_and_measured['Var'])
        df_statistics_and_measured["E_plus_2std"] = df_statistics_and_measured['E'] + 2*np.sqrt(df_statistics_and_measured['Var'])

if set_lower_predictions_to_zero:
    if 'E_minus_std' in df_statistics_and_measured.columns:
        df_statistics_and_measured['E_minus_std'] = df_statistics_and_measured['E_minus_std'].apply(lambda x: max(0, x))
    if 'E_minus_2std' in df_statistics_and_measured.columns:
        df_statistics_and_measured['E_minus_2std'] = df_statistics_and_measured['E_minus_2std'].apply(lambda x: max(0, x))
    if 'P10' in df_statistics_and_measured.columns:
        df_statistics_and_measured['P10'] = df_statistics_and_measured['P10'].apply(lambda x: max(0, x))

if set_mean_prediction_to_zero:
    df_statistics_and_measured['E'] = df_statistics_and_measured['E'].apply(lambda x: max(0, x))

df_statistics_and_measured

# df_statistics_and_measured["E_minus_std"] = df_statistics_and_measured["E"] - df_statistics_and_measured["StdDev"]
# df_statistics_and_measured["E_plus_std"] = df_statistics_and_measured["E"] + df_statistics_and_measured["StdDev"]
# df_statistics_and_measured["E_minus_2std"] = df_statistics_and_measured["E"] - 2*df_statistics_and_measured["StdDev"]
# df_statistics_and_measured["E_plus_2std"] = df_statistics_and_measured["E"] + 2*df_statistics_and_measured["StdDev"]

# df_statistics_and_measured['E'] = df_statistics_and_measured['E'].apply(lambda x: max(0, x))
# if 'E_minus_std' in df_statistics_and_measured:
#     df_statistics_and_measured['E_minus_std'] = df_statistics_and_measured['E_minus_std'].apply(lambda x: max(0, x))
# if 'E_minus_2std' in df_statistics_and_measured:
#     df_statistics_and_measured['E_minus_2std'] = df_statistics_and_measured['E_minus_2std'].apply(lambda x: max(0, x))
# if 'P10' in df_statistics_and_measured:
#     df_statistics_and_measured['P10'] = df_statistics_and_measured['P10'].apply(lambda x: max(0, x))

In [ ]:
dict_what_to_plot = {
    "E_minus_std": False, "E_plus_std": False, 
    "E_minus_2std": True, "E_plus_2std": True,
    "P10": False, "P90": False,
    "StdDev": True, "Skew": False, "Kurt": False, "Sobol_m": False, "Sobol_m2": False, "Sobol_t": False
        }

In [ ]:
# or more detailed plotting of precipitation and temperature as main input data
# predicted streamflow and measured one
# and state data...
for single_qoi in statisticsObject.list_qoi_column:
    df_statistics_and_measured_single_qoi_subset = df_statistics_and_measured.loc[
        df_statistics_and_measured['qoi'] == single_qoi]
    fig = uqef_dynamic_utils.plotting_function_single_qoi(
        df_statistics_and_measured_single_qoi_subset, 
        single_qoi=single_qoi,  
        dict_what_to_plot=dict_what_to_plot,
        directory=directory_for_saving_plots,
        fileName=f"simulation_big_plot_{single_qoi}_2std.html"
    )
#     fig.update_layout(marker_color='blue', row=1, col=1)
#     fig.update_layout(color_discrete_sequence=px.colors.qualitative.G10)

    fig.update_layout(title=None)
    fig.update_layout(
        margin=dict(
            t=10,  # Top margin
            b=10,  # Bottom margin
            l=20,  # Left margin
            r=20   # Right margin
        )
    )
    plot_filename = directory_for_saving_plots / f"simulation_big_plot_{single_qoi}_2std.pdf"
    fig.write_image(str(plot_filename), format="pdf", width=1200,) #height=1000, width=1100,
    
    fig.show()

In [ ]:
# or more detailed plotting of precipitation and temperature as main input data
# predicted streamflow and measured one
# and state data...
for single_qoi in statisticsObject.list_qoi_column:
    df_statistics_and_measured_single_qoi_subset = df_statistics_and_measured.loc[
        df_statistics_and_measured['qoi'] == single_qoi]
    fig = uqef_dynamic_utils.plotting_function_single_qoi_hbv(
        df_statistics_and_measured_single_qoi_subset, 
        single_qoi=single_qoi,  
        qoi=statisticsObject.qoi,
        dict_what_to_plot=dict_what_to_plot,
        directory=directory_for_saving_plots,
        fileName=f"simulation_big_plot_{single_qoi}_2std.html"
    )
#     fig.update_layout(marker_color='blue', row=1, col=1)
#     fig.update_layout(color_discrete_sequence=px.colors.qualitative.G10)

    fig.update_layout(title=None)
    fig.update_layout(
        margin=dict(
            t=10,  # Top margin
            b=10,  # Bottom margin
            l=20,  # Left margin
            r=20   # Right margin
        )
    )
    plot_filename = directory_for_saving_plots / f"simulation_big_plot_{single_qoi}_2std.pdf"
    fig.write_image(str(plot_filename), format="pdf", width=1200,) #height=1000, width=1100,
    
    fig.show()

In [ ]:
fig, _ = uqef_dynamic_utils.plot_forcing_mean_predicted_and_observed_all_qoi(
    statisticsObject, directory=directory_for_saving_plots, fileName="forcing_measured_and_mean_data.html")
fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=20,  # Left margin
        r=20   # Right margin
    )
)
plot_filename = directory_for_saving_plots / f"forcing_measured_and_mean_data.pdf"
fig.write_image(str(plot_filename), format="pdf", height=1000, width=1100,)

fig.show()


In [ ]:
statisticsObject.prepare_for_plotting(
    plot_measured_timeseries=True, 
    plot_forcing_timeseries=True,
    time_column_name="TimeStamp"
)

# single_qoi = statisticsObject.list_qoi_column[0]
for single_qoi in statisticsObject.list_qoi_column:
    statisticsObject.plotResults_single_qoi(
        directory = directory_for_saving_plots,
        fileName = f"{single_qoi}_STAT_SI_TimeSignals",
        display=True, 
        dict_time_vs_qoi_stat=None, 
        single_qoi_column=single_qoi, 
        precipitation_df_timestamp_column="index", 
        temperature_df_timestamp_column="index", 
        streamflow_df_timestamp_column="index",
        dict_what_to_plot=dict_what_to_plot
    )

### Analzing Skew and Kurt in this setup...

In [ ]:
if 'Skew' in df_statistics_and_measured:
    df_statistics_and_measured['Skew'].plot(kind='hist')

In [ ]:
print(df_statistics_and_measured['Skew'].mean())
print(df_statistics_and_measured['Kurt'].mean())


In [ ]:
if 'Kurt' in df_statistics_and_measured:
    df_statistics_and_measured['Kurt'].plot(kind='hist') #'kde'

# GoFs / Metrices / P&R Factors

In [ ]:
qoi_column="Q_cms"

In [ ]:
# df_statistics_and_measured
# uqef_dynamic_utils.compute_gof_over_different_time_series(
#     statisticsObject.df_statistics, 
#     objective_function=["MAE", "NSE", "LogNSE", "RMSE", "NRMSE", "KGE"], 
#     qoi_column=qoi_column, 
#     measuredDF_column_names=["measured"]
# )

statisticsObject.compute_gof_over_different_time_series_single_qoi(
    objective_function=statisticsObject.objective_function, qoi_column=qoi_column)

In [ ]:
p=statisticsObject.calculate_p_factor_single_qoi(
    qoi_column=qoi_column, df_statistics=df_statistics_and_measured,
    column_lower_uncertainty_bound="P10", column_upper_uncertainty_bound="P90",
    observed_column="measured")

In [ ]:
mean_uncertainty_band, std_uncertainty_band, mean_observed, std_observed = statisticsObject.compute_stat_of_uncertainty_band(
    qoi_column=qoi_column, df_statistics=df_statistics_and_measured,
    column_lower_uncertainty_bound="P10", column_upper_uncertainty_bound="P90",
    observed_column="measured"
)

# Sensitivity Analysis - Computing DataFrame with SI and Plotting

In [ ]:
if uqsim_args_dict["compute_Sobol_m"]:
    si_m_df = statisticsObject.create_df_from_sensitivity_indices(si_type="Sobol_m")
else:
    si_m_df = None
si_m_df

In [ ]:
if uqsim_args_dict["compute_Sobol_t"]:
    si_t_df = statisticsObject.create_df_from_sensitivity_indices(si_type="Sobol_t")
else:
    si_t_df = None
si_t_df

In [ ]:
# si_df.set_index(statisticsObject.time_column_name, inplace=True)
for single_qoi in statisticsObject.list_qoi_column:
    fig = statisticsObject.plot_heatmap_si_single_qoi(
        qoi_column=single_qoi, si_df=si_m_df, si_type="Sobol_m")
    fig.update_layout(title_text=f"Sobol First-order SI w.r.t. QoI - {single_qoi}")
    fileName = str(directory_for_saving_plots) + f"Sobol_First_HeatMap_{single_qoi}.html"
    pyo.plot(fig, filename=fileName)
    fig.update_layout(title=None)
    fig.update_layout(
        margin=dict(
            t=10,  # Top margin
            b=10,  # Bottom margin
            l=20,  # Left margin
            r=20   # Right margin
        )
    )
    plot_filename = directory_for_saving_plots / f"sobol_first_heatmap_{single_qoi}.pdf"
    fig.write_image(str(plot_filename), format="pdf", width=1100,)  #height=1000, width=1100,
    fig.show()

In [ ]:
# si_df.set_index(statisticsObject.time_column_name, inplace=True)

si_m_df = si_m_df.sort_values(by=statisticsObject.time_column_name)
si_columns_to_modify = [
    x for x in si_m_df.columns.tolist() if x != 'measured' and x != 'measured_norm' \
    and x != 'qoi' and  x != statisticsObject.time_column_name]
for single_si_columns_to_modify in si_columns_to_modify:
    if single_si_columns_to_modify in si_m_df:
        si_m_df[single_si_columns_to_modify] = si_m_df[single_si_columns_to_modify].apply(lambda x: max(0, x))

for single_qoi in statisticsObject.list_qoi_column:
    fig = statisticsObject.plot_heatmap_si_single_qoi(
        qoi_column=single_qoi, si_df=si_m_df, si_type="Sobol_m")
    fig.update_layout(title_text=f"Sobol First-order SI w.r.t. QoI - {single_qoi}")
    fileName = str(directory_for_saving_plots) + f"Sobol_First_HeatMap_{single_qoi}_set_to_zero.html"
    pyo.plot(fig, filename=fileName)
    fig.update_layout(title=None)
    fig.update_layout(
        margin=dict(
            t=10,  # Top margin
            b=10,  # Bottom margin
            l=20,  # Left margin
            r=20   # Right margin
        )
    )
    plot_filename = directory_for_saving_plots / f"sobol_first_heatmap_{single_qoi}_set_to_zero.pdf"
    fig.write_image(str(plot_filename), format="pdf", width=1100,)  #height=1000, width=1100,
    fig.show()

In [ ]:
for single_qoi in statisticsObject.list_qoi_column:
    fig = statisticsObject.plot_heatmap_si_single_qoi(
        qoi_column=single_qoi, si_df=None, si_type="Sobol_t")
    fig.update_layout(title_text=f"Sobol Total SI w.r.t. QoI - {single_qoi}")
    fileName = str(directory_for_saving_plots) + f"Sobol_Total_HeatMap_{single_qoi}.html"
    pyo.plot(fig, filename=fileName)
    fig.show()

In [ ]:
for single_qoi in statisticsObject.list_qoi_column:
    fig = statisticsObject.plot_si_and_normalized_measured_time_signal_single_qoi(
        qoi_column=single_qoi, si_df=si_m_df, si_type="Sobol_m")
    fig.update_layout(title_text=f"Sobol First SI w.r.t. QoI - {single_qoi}")
    fileName = str(directory_for_saving_plots) + f"Sobol_First_Time_Signal_{single_qoi}_set_to_zero.html"
    # Update x-axis for more dense date display
    # fig.update_xaxes(
    #     tickformat='%b %y',            # Format dates as "Month Day" (e.g., "Jan 01")
    #     dtick="M1"                     # Set tick interval to 1 day for denser ticks
    # )
    pyo.plot(fig, filename=fileName)
    fig.update_layout(title=None)
    fig.update_layout(
        margin=dict(
            t=10,  # Top margin
            b=10,  # Bottom margin
            l=20,  # Left margin
            r=20   # Right margin
        )
    )
    plot_filename = directory_for_saving_plots / f"sobol_first_time_signal_{single_qoi}_set_to_zero.pdf"
    fig.write_image(str(plot_filename), format="pdf", width=1100,)  #height=1000, width=1100,
    fig.show()

In [ ]:
for single_qoi in statisticsObject.list_qoi_column:
    fig = statisticsObject.plot_si_and_normalized_measured_time_signal_single_qoi(
        qoi_column=single_qoi, si_df=None, si_type="Sobol_t")
    fig.update_layout(title_text=f"Sobol Total SI w.r.t. QoI - {single_qoi}")
    fileName = str(directory_for_saving_plots) + f"Sobol_Total_Time_Signal_{single_qoi}.html"
    pyo.plot(fig, filename=fileName)
    fig.update_layout(title=None)
    fig.update_layout(
        margin=dict(
            t=10,  # Top margin
            b=10,  # Bottom margin
            l=20,  # Left margin
            r=20   # Right margin
        )
    )
    plot_filename = directory_for_saving_plots / f"sobol_total_time_signal_{single_qoi}_set_to_zero.pdf"
    fig.write_image(str(plot_filename), format="pdf", width=1100,)  #height=1000, width=1100,
    fig.show()

In [ ]:
for single_qoi in statisticsObject.list_qoi_column:
    fig = statisticsObject.plot_si_and_normalized_measured_time_signal_single_qoi(
        qoi_column=single_qoi, si_df=None, si_type="Sobol_m")
    fig.update_layout(title_text=f"Sobol First SI w.r.t. QoI - {single_qoi}")
    fileName = str(directory_for_saving_plots) + f"Sobol_First_Time_Signal_{single_qoi}.html"
    pyo.plot(fig, filename=fileName)
    fig.update_layout(title=None)
    fig.update_layout(
        margin=dict(
            t=10,  # Top margin
            b=10,  # Bottom margin
            l=20,  # Left margin
            r=20   # Right margin
        )
    )
    plot_filename = directory_for_saving_plots / f"sobol_first_time_signal_{single_qoi}_measured.pdf"
    fig.write_image(str(plot_filename), format="pdf", width=1100,)  #height=1000, width=1100,
    fig.show()

# Analyzing Sobol SI w.r.t. Measured/Forcing Data...

Here important DataFrames computed so far are:
* df_statistics_and_measured
* statisticsObject.forcing_df 
* statisticsObject.df_measured

In [ ]:
single_qoi = statisticsObject.list_qoi_column[0]
# print(f"single_qoi-{single_qoi}")
# df_statistics_and_measured_single_qoi = df_statistics_and_measured.loc[
#     df_statistics_and_measured['qoi'] == single_qoi
# ]
print(f"single_qoi-{single_qoi}")

In [ ]:
statisticsObject.labels

In [ ]:
# No condition
uqef_dynamic_utils.describe_sensitivity_indices_single_qoi_under_some_condition(
    df_statistics_and_measured, single_qoi=single_qoi, param_names=statisticsObject.labels,
    si_type="Sobol_m", condition_columns=None, condition_value=None, condition_sign="equal",
    list_of_columns_to_keep=["measured", "precipitation", "temperature"]
)
# df_statistics_and_measured[list_of_columns_with_sobol_indices].describe(include=np.number)

In [ ]:
# ETF and beta decreased
uqef_dynamic_utils.describe_sensitivity_indices_single_qoi_under_some_condition(
    df_statistics_and_measured, single_qoi=single_qoi, param_names=statisticsObject.labels,
    si_type="Sobol_m", condition_columns="temperature", condition_value=0.0, 
    condition_sign="greater than",
    list_of_columns_to_keep=["measured", "precipitation", "temperature"]
)

In [ ]:
# ETF and beta decreased; FRAM increased
uqef_dynamic_utils.describe_sensitivity_indices_single_qoi_under_some_condition(
    df_statistics_and_measured, single_qoi=single_qoi, param_names=statisticsObject.labels,
    si_type="Sobol_m", condition_columns="temperature", condition_value=5.0, 
    condition_sign="greater than",
    list_of_columns_to_keep=["measured", "precipitation", "temperature"]
)

In [ ]:
# observation FRAC(?) and K2 become more important...
uqef_dynamic_utils.describe_sensitivity_indices_single_qoi_under_some_condition(
    df_statistics_and_measured, single_qoi=single_qoi, param_names=statisticsObject.labels,
    si_type="Sobol_m", condition_columns="measured", condition_value=5.0, condition_sign="smaller_or_equal",
    list_of_columns_to_keep=["measured", "precipitation", "temperature"]
)

In [ ]:
# observation FRAC and K2 become more important...
uqef_dynamic_utils.describe_sensitivity_indices_single_qoi_under_some_condition(
    df_statistics_and_measured, single_qoi=single_qoi, param_names=statisticsObject.labels,
    si_type="Sobol_m", condition_columns="precipitation", condition_value=0.0, condition_sign="equal",
    list_of_columns_to_keep=["measured", "precipitation", "temperature"]
)

In [ ]:
uqef_dynamic_utils.describe_sensitivity_indices_single_qoi_under_some_condition(
    df_statistics_and_measured, single_qoi=single_qoi, param_names=statisticsObject.labels,
    si_type="Sobol_m", condition_columns="precipitation", condition_value=0.0, 
    condition_sign="greater than",
    list_of_columns_to_keep=["measured", "precipitation", "temperature"]
)

In [ ]:
# observation C0 ETF and beta increases; K2 drastically less important, TT increase
uqef_dynamic_utils.describe_sensitivity_indices_single_qoi_under_some_condition(
    df_statistics_and_measured, single_qoi=single_qoi, param_names=statisticsObject.labels,
    si_type="Sobol_m", condition_columns="measured", condition_value=30.0, 
    condition_sign="greater than or equal",
    list_of_columns_to_keep=["measured", "precipitation", "temperature"]
)

In [ ]:
# ETF and beta decreased
uqef_dynamic_utils.describe_sensitivity_indices_single_qoi_under_some_condition(
    df_statistics_and_measured, single_qoi=single_qoi, param_names=statisticsObject.labels,
    si_type="Sobol_m", condition_columns="measured", condition_value=30.0, 
    condition_sign="less than",
    list_of_columns_to_keep=["measured", "precipitation", "temperature"]
)

In [ ]:
corr_df_statistics_and_measured_single_qoi_subset, fig = uqef_dynamic_utils.compute_df_statistics_columns_correlation(
    df_statistics_and_measured, single_qoi=single_qoi, param_names=statisticsObject.labels,
    only_sensitivity_indices_columns=True, si_type="Sobol_m", plot=True,
    list_of_columns_to_keep=["measured", "precipitation", "temperature"]
)

# fig.update_layout(title=None)
# fig.update_layout(
#     margin=dict(
#         t=10,  # Top margin
#         b=10,  # Bottom margin
#         l=20,  # Left margin
#         r=20   # Right margin
#     )
# )
# plot_filename = directory_for_saving_plots / f"sobol_first_time_signal_{single_qoi}_measured.pdf"
# fig.write_image(str(plot_filename), format="pdf", width=1100,)  #height=1000, width=1100,
# fig.show()

In [ ]:
# uqef_dynamic_utils.plot_parameters_sensitivity_indices_vs_temp_prec_measured(
#     df_statistics_and_measured, single_qoi=single_qoi, 
#     param_names=statisticsObject.labels, si_type="Sobol_t",
# )
si_type = "Sobol_m"
fig = uqef_dynamic_utils.plot_parameters_sensitivity_indices_vs_temp_prec_measured_plotly(
    df=df_statistics_and_measured, 
    single_qoi=single_qoi, 
    param_names=statisticsObject.labels, 
    forcing_measured_columns = ["measured", "precipitation", "temperature"],
    si_type="Sobol_m",
)
fig.update_layout(title=f"UQEF-Dynamic Scatter Plot -{si_type}", height=800, width=1000, showlegend=False)

fig.update_layout(title=None)
fig.update_layout(
    margin=dict(
        t=10,  # Top margin
        b=10,  # Bottom margin
        l=20,  # Left margin
        r=20   # Right margin
    )
)
plot_filename = directory_for_saving_plots / f"sobol_m_vs_measured_data_{single_qoi}.pdf"
fig.write_image(str(plot_filename), format="pdf", width=1100,)  #height=1000, width=1100,

fig.show()

In [ ]:
fig = uqef_dynamic_utils.plot_parameters_sensitivity_indices_vs_temp_prec_measured(
    df_statistics_and_measured, single_qoi=single_qoi, 
    param_names=statisticsObject.labels, si_type="Sobol_m",
)
# fig.update_layout(title=None)
# fig.update_layout(
#     margin=dict(
#         t=10,  # Top margin
#         b=10,  # Bottom margin
#         l=20,  # Left margin
#         r=20   # Right margin
#     )
# )
filename = directory_for_saving_plots / f"sobol_m_vs_measured_data_{single_qoi}.pdf"
fig.savefig(str(filename), format="pdf")


In [ ]:
uqef_dynamic_utils.plot_cdfs_of_parameters_sensitivity_indices(
    df_statistics_and_measured, single_qoi=single_qoi, 
    param_names=statisticsObject.labels, si_type="Sobol_t")

In [ ]:
uqef_dynamic_utils.single_param_single_qoi_sensitivity_indices_GaussianKDE(
    df_statistics_and_measured, single_qoi=single_qoi, 
    param_name="TT", si_type="Sobol_t", plot=True, plot_pdf_or_cdf="pdf")

In [ ]:
uqef_dynamic_utils.single_param_single_qoi_sensitivity_indices_GaussianKDE(
    df_statistics_and_measured, single_qoi=single_qoi, 
    param_name="TT", si_type="Sobol_t", plot=True, plot_pdf_or_cdf="pdf")

In [ ]:
uqef_dynamic_utils.single_param_single_qoi_sensitivity_indices_GaussianKDE(
    df_statistics_and_measured, single_qoi=single_qoi, 
    param_name="TT", si_type="Sobol_t", plot=True, plot_pdf_or_cdf="cdf")

# gPCE Surrogate

## The analysis is presented only for the first QoI...
If computed...

In [ ]:
gPCE_model = defaultdict()
for single_date in statisticsObject.pdTimesteps:
    gPCE_model[single_date] = statisticsObject.result_dict[statisticsObject.list_qoi_column[0]][single_date]['gPCE']

In [ ]:
simulationNodes.joinedStandardDists

In [ ]:
simulationNodes.joinedDists

In [ ]:
samples_to_evaluate_gPCE = simulationNodes.joinedDists.sample(100, rule="random") # 'sobol' 'random'
samples_to_evaluate_gPCE_transformed = utility.transformation_of_parameters_var1(
    samples_to_evaluate_gPCE, simulationNodes.joinedDists, simulationNodes.joinedStandardDists)

In [ ]:
samples_to_evaluate_gPCE_transformed.shape

## Re-evaluate gPCE surrogate for single date on a smaller portion on data

In [ ]:
temp_pdTimesteps = statisticsObject.pdTimesteps[0:2]
single_date = temp_pdTimesteps[0]
gPCE_model_sinlge_date = statisticsObject.result_dict[statisticsObject.list_qoi_column[0]][single_date]['gPCE']

In [ ]:
result = gPCE_model_sinlge_date(*samples_to_evaluate_gPCE_transformed)

In [ ]:
result.shape

## Evaluating the gPCE Surrogate for all the time-steps - Note: this might be computationally expensive!

In [ ]:
start = time.time()

gPCE_model_evaluated = defaultdict()
for single_date in statisticsObject.pdTimesteps:
    gPCE_model_single_date = statisticsObject.result_dict[statisticsObject.list_qoi_column[0]][single_date]['gPCE']
    gPCE_model_evaluated[single_date] = gPCE_model_single_date(samples_to_evaluate_gPCE_transformed.T)

end = time.time()
runtime = end - start
print(f"Time needed for evaluating {samples_to_evaluate_gPCE_transformed.shape[1]} \
gPCE model for {len(statisticsObject.pdTimesteps)} days is: {runtime}")

In [ ]:
gpc_eval_df = pd.DataFrame.from_dict(gPCE_model_evaluated, orient="index", columns=range(1000))
gpc_eval_df

In [ ]:
gpc_eval_df['new_E'] = gpc_eval_df.mean(numeric_only=True, axis=1)
gpc_eval_df = gpc_eval_df.loc[:, ['new_E',]]
gpc_eval_df

## Looking closer at the values of the gPCE surrogate for some specific time (i.e., single date)

In [ ]:
# how many coefficints to expect...
print(uqsim_args_dict['sc_q_order'])
print(uqsim_args_dict['sc_p_order'])
print(scipy.special.binom(9, 3))

In [ ]:
date_QoI = pd.Timestamp('2007-06-16 00:00:00')
type(gPCE_model[date_QoI])

In [ ]:
# sorting based on coefficient values...
sorted(gPCE_model[date_QoI].coefficients)

gPCE_model_single_date_dict = gPCE_model[date_QoI].todict()

gPCE_model_single_date_dict_sorted = dict(sorted(gPCE_model_single_date_dict.items(), key=lambda item: item[1]))
gPCE_model_single_date_dict_sorted

In [ ]:
for key, value in gPCE_model_single_date_dict_sorted.items():
    if abs(value) > 0.5:
        print(f"{key}: {value}")

In [ ]:
l2 = cp.outer(gPCE_model[date_QoI] , gPCE_model[date_QoI] )
norm = cp.E(l2, simulationNodes.joinedStandardDists).round(15)
norm

In [ ]:
float(cp.E(gPCE_model[date_QoI], simulationNodes.joinedStandardDists))

In [ ]:
import numpoly
# re-creating a new poly based on ...
poly_new = numpoly.set_dimensions(gPCE_model[date_QoI], len(simulationNodes.joinedStandardDists))

In [ ]:
gPCE_model[date_QoI].exponents.T

In [ ]:
moments = simulationNodes.joinedStandardDists.mom(gPCE_model[date_QoI].exponents.T)
moments

In [ ]:
len(moments)

In [ ]:
out = np.zeros(gPCE_model[date_QoI].shape)
for idx, key in enumerate(gPCE_model[date_QoI].keys):
    out += gPCE_model[date_QoI].values[key] * moments[idx]
out

# Analyzing gPCE coeff. values accross all the dates - computing mean over coefficients, and difference btw different gPCEs, visualization...

In [ ]:
date_QoI = statisticsObject.pdTimesteps[0]
# polynomial_indices = list(gPCE_model[date_QoI].todict().keys())

mean_coefieinct_of_polynomials = defaultdict(float, {k:0. for k in gPCE_model[date_QoI].todict().keys()})
    
percantege_greater_than_dict = dict()
for date in statisticsObject.pdTimesteps:
#     current_gPCE_surrogate_sorted = dict(sorted(gPCE_model[date].todict().items(), key=lambda item: item[1]))
    percantege_greater_than = 0
    counter = 0
    for key, value in gPCE_model[date].todict().items():
        counter += 1
        mean_coefieinct_of_polynomials[key] = mean_coefieinct_of_polynomials[key] + value
        if abs(value) > 0.5:
            percantege_greater_than +=1
    percantege_greater_than = percantege_greater_than/counter
    percantege_greater_than = percantege_greater_than*100
    percantege_greater_than_dict[date] = percantege_greater_than
    print(f"for date - {date} perc. of values grater than 0.5 is {percantege_greater_than} %")
    
N = len(statisticsObject.pdTimesteps)
for key in mean_coefieinct_of_polynomials.keys():
    mean_coefieinct_of_polynomials[key] = mean_coefieinct_of_polynomials[key]/N
 
# mean_coefieinct_of_polynomials_sorted = dict(sorted(mean_coefieinct_of_polynomials.items(), key=lambda item: item[1]))
mean_coefieinct_of_polynomials_sorted = mean_coefieinct_of_polynomials

In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=list(percantege_greater_than_dict.keys()), 
    y=list(percantege_greater_than_dict.values()))
             )
fig.show()

In [ ]:
mean_coefieinct_of_polynomials_sorted_df = pd.DataFrame(
    mean_coefieinct_of_polynomials_sorted.items(), columns=['poly_index', 'coeff'])
mean_coefieinct_of_polynomials_sorted_df = mean_coefieinct_of_polynomials_sorted_df.astype({'poly_index': "category"})
mean_coefieinct_of_polynomials_sorted_df['poly_index'] = mean_coefieinct_of_polynomials_sorted_df['poly_index'].apply(lambda x: str(x))


In [ ]:
mean_coefieinct_of_polynomials_sorted_df.dtypes

In [ ]:
date_QoI = pd.Timestamp('2007-06-16 00:00:00')

In [ ]:
date_QoI_05_01 = pd.Timestamp('2007-05-01 00:00:00')
gPCE_model_single_date = gPCE_model[date_QoI_05_01]
gPCE_model_single_date_df = pd.DataFrame(
    gPCE_model_single_date.todict().items(), columns=['poly_index', 'coeff'])
gPCE_model_single_date_df_05_01 = gPCE_model_single_date_df.astype({'poly_index': "category"})
gPCE_model_single_date_df_05_01['poly_index'] = gPCE_model_single_date_df_05_01['poly_index'].apply(lambda x: str(x))

# merge with the mean
# df_statistics_and_measured = pd.merge(
#     statisticsObject.df_statistics, statisticsObject.forcing_df, left_on=statisticsObject.time_column_name, right_index=True)

In [ ]:
date_QoI_06_16 = pd.Timestamp('2007-06-16 00:00:00')
gPCE_model_single_date = gPCE_model[date_QoI_06_16]
gPCE_model_single_date_df = pd.DataFrame(
    gPCE_model_single_date.todict().items(), columns=['poly_index', 'coeff'])
gPCE_model_single_date_df_06_16 = gPCE_model_single_date_df.astype({'poly_index': "category"})
gPCE_model_single_date_df_06_16['poly_index'] = gPCE_model_single_date_df_06_16['poly_index'].apply(lambda x: str(x))



In [ ]:
date_QoI_09_01 = pd.Timestamp('2007-09-01 00:00:00')
gPCE_model_single_date = gPCE_model[date_QoI_09_01]
gPCE_model_single_date_df = pd.DataFrame(
    gPCE_model_single_date.todict().items(), columns=['poly_index', 'coeff'])
gPCE_model_single_date_df_09_01 = gPCE_model_single_date_df.astype({'poly_index': "category"})
gPCE_model_single_date_df_09_01['poly_index'] = gPCE_model_single_date_df_09_01['poly_index'].apply(lambda x: str(x))



In [ ]:
fig = go.Figure()
fig.add_trace(go.Bar(
    x=mean_coefieinct_of_polynomials_sorted_df.poly_index,
    y=mean_coefieinct_of_polynomials_sorted_df.coeff,
    name='Mean gPCE',
    marker_color='MediumPurple'
))

fig.add_trace(go.Bar(
    x=gPCE_model_single_date_df_05_01.poly_index,
    y=gPCE_model_single_date_df_05_01.coeff,
    name=f'date {date_QoI_05_01}',
    marker_color='ForestGreen'
))
fig.add_trace(go.Bar(
    x=gPCE_model_single_date_df_06_16.poly_index,
    y=gPCE_model_single_date_df_06_16.coeff,
    name=f'date {date_QoI_06_16}',
    marker_color='Tomato'
))
fig.add_trace(go.Bar(
    x=gPCE_model_single_date_df_09_01.poly_index,
    y=gPCE_model_single_date_df_09_01.coeff,
    name=f'date {date_QoI_09_01}',
    marker_color='RoyalBlue'
))
# Here we modify the tickangle of the xaxis, resulting in rotated labels.
fig.update_layout(barmode='group')
fileName = str(directory_for_saving_plots) + f"Comparing_gPCE_coeff_for_three_dates.html"
pyo.plot(fig, filename=fileName)
fig.show()